In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from PIL import Image
import base64
import io
import json
from datetime import datetime
import uuid
import time
import re

import brand_director as bd
import nametag_prompts as pn

bd.setup_environment()

# 함수에서 반환된 키를 변수에 저장하여 다른 곳에서 활용
GEMINI_API_KEY, GEMINI_IMAGE_API_KEY = bd.load_gemini_keys()

# 시스템 프롬프트를 변수에 저장하여 다른 곳에서 활용
SYSTEM_PROMPT = pn.SYSTEM_PROMPT

✅ Gemini API 키 로드 완료
✅ Gemini Image API 키 로드 완료


In [ ]:
brand_info = {'brand_name': '시스테마',
 'brand_name_en': 'Systema',
 'name_meaning': '체계와 질서를 뜻하는 라틴어 기반의 네이밍',
 'slogan': '데이터로 증명하는 브랜딩의 정석',
 'story_summary': '흩어진 아이디어를 냉철하게 분석하고 논리적 세포 단위로 재구성하여 설득력 있는 브랜드 결과물을 도출합니다.',
 'seed_color': '#4A4A4A',
 'seed_color_reason': '단단한 금속의 질감을 닮은 냉철한 스틸 그레이'}

interview_data_A = "Q1. 시스테마의 '논리적 세포 단위 재구성'이라는 철학을 시각화할 때, 어떤 그래픽 스타일이 가장 적합하다고 생각하시나요?\nA: 정교하게 설계된 그리드 시스템 기반의 미니멀한 고딕 스타일\n\nQ2. 고객이 브랜드로부터 메시지를 받을 때, 시스테마의 페르소나는 어떤 화법을 사용해야 할까요?\nA: 시장의 비효율을 비판하며 명확한 정답을 제시하는 권위적인 어조\n\nQ3. 고객이 시스테마를 처음 접하는 순간, 브랜드의 '정밀함'을 체감할 결정적인 경험은 무엇이어야 할까요?\nA: 모든 정보가 논리적으로 정렬되어 탐색이 극도로 편리한 웹사이트\n\nQ4. 시스테마가 시장에 가장 먼저 출시하여 가치를 증명할 '히어로 제품/서비스'의 형태는 무엇입니까?\nA: 데이터 분석 기반의 브랜드 전략 컨설팅 서비스"interview_data_B = "Q1. 숲결의 페르소나가 평일 일과를 마치고 현관문을 열었을 때, 가장 먼저 해소하고 싶어 하는 감각적 '결핍'은 무엇인가요?\nA: 자아의 회복: 사회적 역할에서 벗어나 오직 '나의 취항'으로만 채워진 안식\n\nQ2. 이 고객이 숲결의 제품을 구매하며 느끼고 싶어 하는 '가장 결정적인 심리적 트리거'는 무엇인가요?\nA: 취향의 차별화: 흔하지 않은 감도의 브랜드를 발견하고 향유하는 미적 우월감\n\nQ3. 이 타겟이 숲결이라는 브랜드를 처음 발견하게 될 가능성이 가장 높은 경로는 무엇인가요?\nA: 알고리즘 기반 이미지 탐색: 인스타그램이나 핀터레스트의 감각적인 인테리어 무드보드"
interview_data_B = "Q1. 이 타겟이 침대에 누워 잠들기 직전, 오늘 업무나 일상에서 가장 크게 느꼈을 '결핍'이나 스트레스는 무엇일까요?\nA: 주관적인 취향과 감정에 치우친 의사결정들 사이에서 명확한 기준점을 찾지 못한 혼란\n\nQ2. 이 타겟이 '시스테마'의 서비스를 최종 결제하게 만드는 결정적인 '심리적 트리거'는 무엇일까요?\nA: 복잡한 문제를 한 단어로 관통하는 명쾌한 질서를 목격했을 때 느끼는 카타르시스\n\nQ3. 이 타겟이 평소 자신의 전문성을 강화하거나 고차원적인 정보를 습득하기 위해 가장 자주 머무는 '디지털/물리적 공간'은 어디인가요?\nA: 업계 최고의 전문가들이 모여 논리적인 비판과 토론을 즐기는 폐쇄적인 커뮤니티"

In [3]:
raw_text_C_0 = bd.request_gemini_api(pn.get_C0_visual_interview_prompt(brand_info), SYSTEM_PROMPT)
parsed_response_C_0 = bd.parse_ai_response(raw_text_C_0)


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

✅ AI 응답 수신 완료!
응답 텍스트: {
  "reasoning": "브랜드 네임 '숲결'과 슬로건이 지닌 정서적 가치를 시각적 체계로 구체화하기 위해, 공간의 여백 구성과 글꼴의 표정, 그리고 빛의 운용 방식을 정의...
✅ JSON 파싱 완료!
파싱된 JSON: {"reasoning": "\ube0c\ub79c\ub4dc \ub124\uc784 '\uc232\uacb0'\uacfc \uc2ac\ub85c\uac74\uc774 \uc9c0\ub2cc \uc815\uc11c\uc801 \uac00\uce58\ub97c \uc2dc\uac01\uc801 \uccb4\uacc4\ub85c \uad6c\uccb4\ud654\ud558\uae30 \uc704\ud574, \uacf5\uac04\uc758 \uc5ec\ubc31 \uad6c\uc131\uacfc \uae00\uaf34\uc758 \ud45c\uc815, \uadf8\ub9ac\uace0 \ube5b\uc758 \uc6b4\uc6a9 \ubc29\uc2dd\uc744 \uc815\uc758\ud558\uace0\uc790 \ud569\ub2c8\ub2e4. \uc774\ub97c \ud1b5\ud574 \ucc3d\uc5c5\uc790\uac00 \uc9c0\ud5a5\ud558\ub294 '\uc628\ub3c4'\uac00 \ubbf8\ub2c8\uba40\ud55c \ud604\ub300\uc801 \uac10\uac01\uc778\uc9c0, \ud639\uc740 \uc544\ub0

In [4]:
# AI가 왜 이런 질문을 만들었는지 사용자에게 보여주면 신뢰도가 확 올라갑니다!
print(f"💡 AI 분석: {parsed_response_C_0['reasoning']}\n")

# 사용자의 최종 답변을 모아둘 리스트
collected_answers_C = bd.conduct_ai_interview(parsed_response_C_0, title="🤖 Section C 비주얼 아이덴티티 인터뷰")
interview_data_C = bd.format_interview_responses(collected_answers_C)

💡 AI 분석: 브랜드 네임 '숲결'과 슬로건이 지닌 정서적 가치를 시각적 체계로 구체화하기 위해, 공간의 여백 구성과 글꼴의 표정, 그리고 빛의 운용 방식을 정의하고자 합니다. 이를 통해 창업자가 지향하는 '온도'가 미니멀한 현대적 감각인지, 혹은 아날로그적인 서정성인지 명확히 구분하여 독보적인 비주얼 가이드를 구축할 것입니다.


🤖 Section C 비주얼 아이덴티티 인터뷰
💡 AI 분석: 브랜드 네임 '숲결'과 슬로건이 지닌 정서적 가치를 시각적 체계로 구체화하기 위해, 공간의 여백 구성과 글꼴의 표정, 그리고 빛의 운용 방식을 정의하고자 합니다. 이를 통해 창업자가 지향하는 '온도'가 미니멀한 현대적 감각인지, 혹은 아날로그적인 서정성인지 명확히 구분하여 독보적인 비주얼 가이드를 구축할 것입니다.


[질문 1/3. 첫 번째 질문: 브랜드의 첫인상을 결정짓는 전체적인 형태와 레이아웃은 어떤 느낌을 지향하시나요?]
  1. 광활한 여백을 활용하여 숨통이 트이는 듯한 극도의 미니멀리즘 레이아웃
  2. 비정형적인 곡선과 유기적인 형태를 활용한 부드럽고 자연스러운 흐름의 구성
  3. 반듯한 직선과 격자(Grid) 시스템을 활용하여 신뢰감을 주는 정갈하고 체계적인 구조
  4. 다양한 질감의 요소들을 겹치고 배치하여 따뜻한 온기가 가득 채워진 풍성한 스타일

[질문 2/3. 두 번째 질문: 브랜드의 목소리가 되어줄 글꼴(Typography)은 어떤 표정을 짓고 있어야 할까요?]
  1. 섬세하고 우아한 획의 끝처리가 돋보이는 서정적인 명조체(Serif) 계열
  2. 군더더기 없이 깔끔하고 정직한 인상을 주는 현대적인 고딕체(Sans-serif) 계열
  3. 제작자의 손길과 온기가 직접적으로 전달되는 듯한 자유로운 핸드라이팅(Handwritten) 스타일
  4. 전통적인 붓 터치의 질감이 살아있어 깊이감과 예스러움이 느껴지는 서예 스타일

[질문 3/3. 세 번째 질문: 제품과 브랜드를 담아낼 사진의 '빛과 공기감'은 어떤 온도를 머금고 있어야 할까요?

In [5]:
raw_text_C_1 = bd.request_gemini_api(pn.get_C1_visual_identity_prompt(brand_info, f"{interview_data_A} + {interview_data_B}", interview_data_C), SYSTEM_PROMPT)
parsed_response_C_1 = bd.parse_ai_response(raw_text_C_1)


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

✅ AI 응답 수신 완료!
응답 텍스트: {
  "color_palette": [
    {
      "color_name": "깊은 숲의 잔영",
      "hex_code": "#2C3E2D",
      "rol...
✅ JSON 파싱 완료!
파싱된 JSON: {"color_palette": [{"color_name": "\uae4a\uc740 \uc232\uc758 \uc794\uc601", "hex_code": "#2C3E2D", "role": "Primary", "reason": "\ube0c\ub79c\ub4dc\uba85 '\uc232'\uc744 \uc0c1\uc9d5\ud558\uba70 \ud050\ub808\uc774\ud130\uc758 \uc804\ubb38\uc131\uacfc \uc2e0\ub8b0\uac10\uc744 \ubd80\uc5ec\ud558\ub294 \uc9d9\uc740 \uadf8\ub9b0"}, {"color_name": "\uc548\uac1c \ub080 \uc5ec\uba85", "hex_code": "#A0AAB2", "role": "Secondary", "reason": "\uc2ec\uce35 \uc778\ud130\ubdf0\uc5d0\uc11c \uc5b8\uae09\ub41c \uc0c8\ubcbd \uc232\uc758 \ucc28\ubd84\ud558\uace0 \uba85\uc0c1\uc801\uc778 \ubd84\uc704\uae30\ub97c \uc2dc\uac01\ud65

In [6]:
parsed_response_C_1

{'color_palette': [{'color_name': '깊은 숲의 잔영',
   'hex_code': '#2C3E2D',
   'role': 'Primary',
   'reason': "브랜드명 '숲'을 상징하며 큐레이터의 전문성과 신뢰감을 부여하는 짙은 그린"},
  {'color_name': '안개 낀 여명',
   'hex_code': '#A0AAB2',
   'role': 'Secondary',
   'reason': '심층 인터뷰에서 언급된 새벽 숲의 차분하고 명상적인 분위기를 시각화'},
  {'color_name': '투명한 결',
   'hex_code': '#D9E4E1',
   'role': 'Secondary',
   'reason': '업사이클링 유리의 영롱함과 투명한 소재감을 보조하는 맑은 그레이쉬 블루'},
  {'color_name': '윤슬의 반짝임',
   'hex_code': '#C49A6C',
   'role': 'Accent',
   'reason': '햇살의 반짝임과 고전적 미감을 강조하여 취향의 차별화를 완성하는 포인트'},
  {'color_name': '새벽의 모래알',
   'hex_code': '#F5E6D3',
   'role': 'Background',
   'reason': '사용자 선택 시드 컬러로, 전체 브랜드에 따뜻한 안식의 온도를 형성하는 바탕색'}],
 'typography': {'primary_font': {'category': '헤드라인 및 타이틀',
   'font_name_kr': '송명',
   'google_fonts_family': 'Song Myung',
   'weight_recommendation': 400,
   'usage_and_reason': '전통적 붓 터치의 질감과 고전적인 세리프가 조화를 이루어 미적 우월감을 표현'},
  'secondary_font': {'category': '본문 및 일반 텍스트',
   'font_name_kr': '본고딕',
   'goo